# Distance pipeline notebook

This notebook mirrors the refactored pipeline while keeping the intermediate artefacts easy to inspect.

In [ ]:
from pathlib import Path
import sys

import geopandas as gpd
import pandas as pd

In [ ]:
PROJECT_ROOT = Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

In [ ]:
from distance_pipeline.cache import CacheManager
from distance_pipeline.candidate_builder import build_candidate_sites
from distance_pipeline.config_loader import load_cfg
from distance_pipeline.distance_matrix import compute_distances
from distance_pipeline.facilities import load_health_facilities
from distance_pipeline.io import download_file
from distance_pipeline.network import build_pandana_network, load_osm_network
from distance_pipeline.pipeline_support import build_context_map_path, build_map_facilities, build_output_run_tag, resolve_candidate_grid_spacing, resolve_candidate_max_snap_dist
from distance_pipeline.population import worldpop_to_points
from distance_pipeline.settings import PipelineSettings
from distance_pipeline.snapping import snap_points_to_nodes
from distance_pipeline.source_tables import combine_existing_and_candidate_sources, ensure_id_column, ensure_id_index_matches, set_known_categories
from distance_pipeline.viz import classify_roads, plot_context_map, to_point_geometries

In [ ]:
def format_shape(df: pd.DataFrame) -> str:
    '''Return shape formatted with thousands separators.'''
    rows, cols = df.shape
    return f'{rows:,} x {cols:,}'

In [ ]:
COUNTRY_CODE = 'timor_leste'

settings = PipelineSettings(
    population_threshold=1.0,
    sample_fraction=1.0,
    max_points=None,
    max_total_dist=None,
    candidate_grid_spacing_m=None,  # use cfg default
    candidate_max_snap_dist_m=None,  # use cfg default
    force_recompute=False,
    verbose=True,
    save_context_map=False,
    show_context_map=False,
    context_map_path=None,
    context_map_dpi=300,
)

cfg = load_cfg(COUNTRY_CODE)
agg = cfg.aggregate_factor

cache = CacheManager(
    cfg=cfg,
    force_recompute=settings.force_recompute,
    verbose=settings.verbose,
)

print(cfg)
print(f'Candidate grid spacing from cfg: {cfg.candidate_grid_spacing_m}')
print(f'Candidate max snap distance from cfg: {cfg.candidate_max_snap_dist_m}')

In [ ]:
print(f'Country: {cfg.COUNTRY_NAME}')
print(f'Base directory: {cfg.BASE_DIR}')
print(f'PBF path: {cfg.PBF_PATH}')
print(f'WorldPop path: {cfg.WORLDPOP_PATH}')
print(f'Distance threshold km: {cfg.DISTANCE_THRESHOLD_KM}')
print(f'Projected EPSG: {cfg.PROJECTED_EPSG}')
print()
print(settings)

In [ ]:
cfg.BASE_DIR.mkdir(parents=True, exist_ok=True)

download_file(cfg.PBF_URL, cfg.PBF_PATH, overwrite=False, verbose=settings.verbose)
download_file(cfg.WORLDPOP_URL, cfg.WORLDPOP_PATH, overwrite=False, verbose=settings.verbose)

In [ ]:
nodes, edges = cache.load_or_build_network_data(
    builder=lambda: load_osm_network(cfg.PBF_PATH, verbose=settings.verbose)[1:],
)
network = build_pandana_network(nodes=nodes, edges=edges)

print(f'nodes shape: {format_shape(nodes)}')
print(f'edges shape: {format_shape(edges)}')
nodes.head()

In [ ]:
roads = cache.run(
    cache_path=cache.roads_path(),
    builder=lambda: classify_roads(edges, verbose=settings.verbose),
)
print(f'Classified roads shape: {format_shape(roads)}')
roads.head()

In [ ]:
population_points = cache.run(
    cache_path=cache.population_points_path(
        population_threshold=settings.population_threshold,
        sample_fraction=settings.sample_fraction,
        max_points=settings.max_points,
        aggregate_factor=agg,
    ),
    builder=lambda: worldpop_to_points(
        cfg.WORLDPOP_PATH,
        population_threshold=settings.population_threshold,
        aggregate_factor=agg,
        sample_fraction=settings.sample_fraction,
        max_points=settings.max_points,
        verbose=settings.verbose,
    ),
)

print(format_shape(population_points))
population_points.head()

In [ ]:
amenity_values = None
include_healthcare_tag = True

health_centers = cache.run(
    cache_path=cache.health_facilities_path(
        amenity_values=amenity_values,
        include_healthcare_tag=include_healthcare_tag,
    ),
    builder=lambda: load_health_facilities(
        cfg.PBF_PATH,
        amenity_values=amenity_values,
        include_healthcare_tag=include_healthcare_tag,
        verbose=settings.verbose,
    ),
)

health_centers = cache.run(
    cache_path=cache.health_facilities_points_path(
        amenity_values=amenity_values,
        include_healthcare_tag=include_healthcare_tag,
    ),
    builder=lambda: to_point_geometries(
        health_centers,
        projected_epsg=cfg.PROJECTED_EPSG,
        verbose=settings.verbose,
    ),
)

print(health_centers.shape)
health_centers[['ID', 'Name']].head()

In [ ]:
candidate_sites, candidate_sites_snapped = build_candidate_sites(
    cfg=cfg,
    settings=settings,
    cache=cache,
    nodes=nodes,
)
candidate_grid_spacing_m = resolve_candidate_grid_spacing(cfg, settings)
candidate_max_snap_dist_m = resolve_candidate_max_snap_dist(cfg, settings)


In [ ]:
if settings.show_context_map or settings.save_context_map:
    map_facilities = build_map_facilities(
        health_centers=health_centers,
        candidate_sites=candidate_sites_snapped,
    )

    context_map_path = build_context_map_path(
        default_path=cache.context_map_path(),
        user_path=settings.context_map_path,
        candidate_grid_spacing_m=candidate_grid_spacing_m,
    )

    plot_context_map(
        roads=roads,
        population_points=population_points,
        health_centers=map_facilities,
        title=cfg.PLOT_TITLE,
        output_path=context_map_path if settings.save_context_map else None,
        dpi=settings.context_map_dpi,
        show=settings.show_context_map,
        verbose=settings.verbose,
    )

In [ ]:
population = cache.run(
    cache_path=cache.population_snapped_path_for(
        distance_col='dist_snap_target',
        population_threshold=settings.population_threshold,
        sample_fraction=settings.sample_fraction,
        max_points=settings.max_points,
        aggregate_factor=agg,
    ),
    builder=lambda: snap_points_to_nodes(
        population_points,
        nodes,
        distance_col='dist_snap_target',
        projected_epsg=cfg.PROJECTED_EPSG,
        verbose=settings.verbose,
    ),
)

population = ensure_id_column(population, prefix='target')
population = ensure_id_index_matches(population)

print(format_shape(population))
population.head()

In [ ]:
existing_sources = cache.run(
    cache_path=cache.hospitals_snapped_path_for(
        distance_col='dist_snap_source',
        amenity_values=amenity_values,
        include_healthcare_tag=include_healthcare_tag,
    ),
    builder=lambda: snap_points_to_nodes(
        health_centers,
        nodes,
        distance_col='dist_snap_source',
        projected_epsg=cfg.PROJECTED_EPSG,
        verbose=settings.verbose,
    ),
)

print(existing_sources.shape)
existing_sources.head()

existing_sources = existing_sources.copy()
existing_sources['source_type'] = 'existing'

existing_sources = ensure_id_column(existing_sources, prefix='existing')
existing_sources = ensure_id_index_matches(existing_sources)


In [ ]:
print(population.dtypes)
print()
print(existing_sources.dtypes)

In [ ]:
print(population.columns.tolist())
print(existing_sources.columns.tolist())

assert 'Longitude' in population.columns
assert 'Latitude' in population.columns
assert 'Longitude' in existing_sources.columns
assert 'Latitude' in existing_sources.columns

In [ ]:
sources_for_matrix = combine_existing_and_candidate_sources(
    facilities=existing_sources,
    candidate_sites_snapped=candidate_sites_snapped,
)

matrix_cache_path = cache.distance_matrix_path_for(
    distance_threshold_largest=cfg.DISTANCE_THRESHOLD_KM,
    max_total_dist=settings.max_total_dist,
    population_threshold=settings.population_threshold,
    sample_fraction=settings.sample_fraction,
    max_points=settings.max_points,
    aggregate_factor=agg,
    amenity_values=amenity_values,
    include_healthcare_tag=include_healthcare_tag,
    candidate_grid_spacing_m=candidate_grid_spacing_m,
    candidate_max_snap_dist_m=candidate_max_snap_dist_m,
    has_candidates=candidate_sites_snapped is not None,
)

matrix_df = cache.run(
    cache_path=matrix_cache_path,
    builder=lambda: compute_distances(
        targets=population,
        sources=sources_for_matrix,
        distance_threshold_largest=cfg.DISTANCE_THRESHOLD_KM,
        network=network,
        max_total_dist=settings.max_total_dist,
        verbose=settings.verbose,
    ),
)

matrix_df = set_known_categories(matrix_df)

print(f'rows: {matrix_df.shape[0]:,}, cols: {matrix_df.shape[1]:,}')
matrix_df.head()


In [ ]:
with pd.option_context('display.float_format', '{:,.0f}'.format):
    display(matrix_df['total_dist'].describe())

In [ ]:
output_dir = cfg.BASE_DIR / 'outputs'
output_dir.mkdir(parents=True, exist_ok=True)

run_tag = build_output_run_tag(
    settings=settings,
    aggregate_factor=agg,
    amenity_values=amenity_values,
    include_healthcare_tag=include_healthcare_tag,
    candidate_grid_spacing_m=candidate_grid_spacing_m,
    candidate_max_snap_dist_m=candidate_max_snap_dist_m,
    has_candidates=candidate_sites_snapped is not None,
)

population_path = output_dir / f'population_{run_tag}.parquet'
existing_sources_path = output_dir / f'existing_sources_{run_tag}.parquet'
matrix_path = output_dir / f'distance_matrix_{run_tag}.parquet'

population.to_parquet(population_path, index=False)
existing_sources.to_parquet(existing_sources_path, index=False)
matrix_df.to_parquet(matrix_path, index=False)

print(population_path)
print(existing_sources_path)
print(matrix_path)

In [ ]:
print(f'Population rows: {len(population):,}')
print(f'Existing source rows: {len(existing_sources):,}')
print(f'Distance rows: {len(matrix_df):,}')

print()
print(f'Unique population ids in matrix: {matrix_df['target_id'].nunique():,}')
print(f'Unique source ids in matrix: {matrix_df['source_id'].nunique():,}')

In [ ]:
dists = {c : matrix_df[c].sum() for c in matrix_df.columns if 'dist' in c}

In [ ]:
with pd.option_context('display.float_format', '{:,.2f}'.format):
    display(matrix_df[dists.keys()].describe())

In [ ]:
{
    key: f'{value / dists["total_dist"]:.2%}'
    for key, value in dists.items()
    if key != 'total_dist'
}